## **Step by step impletation of Service RAG :**
#### Step 1: Load pymupdf and load pdf

In [3]:
import os
from pathlib import Path

# Walk up until we find the project root (folder containing 'data')
def find_project_root(marker="data"):
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    return p

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Files here:", [f.name for f in PROJECT_ROOT.iterdir()])

Project root: c:\Users\DELL\OneDrive\Documents\Amazn-AI
Files here: ['.env', '.git', '.gitignore', '.venv', 'app.py', 'Assets', 'data', 'LICENSE', 'notebooks', 'README.md', 'requirements.txt', 'src', 'test.py', 'vector_store']


In [8]:
import fitz
doc=fitz.open("data/raw/Amazon-Support.pdf")
pages=[]
for page_number ,page in enumerate(doc,start=1):
    page_text=page.get_text()
    pages.append({
        "page":page_number,
        "text":page_text
    })

doc.close()
print(pages[0]["text"])

AMAZON.COM — HELP CENTRE
Amazon Support
Knowledge Base
Orders • Delivery • Returns • Refunds • Payments • Prime • Account & Privacy
20 sections · 359 answered entries · 22 reference tables · worked case files · letter templates · topic index
Search the help centre:  where is my order  |  return window  |  refund not received  |  a-to-z claim
 |  prime renewal
Edition
Compiled edition 2026.09 — Rev. 3.0
Coverage
US consumer and business accounts, with an India comparison in Appendix C and a
seller-side section
Compiled
Last reviewed: 30 September 2026
Classification
Desk reference — buyer, seller and support trainee edition
AMZ-KB-2026-09 · REV 3.0 · 20 SECTIONS · 359 ENTRIES
NOTICE. This is an independently compiled reference guide written in the style of an e-commerce help centre. It is not
produced, reviewed or endorsed by Amazon.com, Inc. or any affiliate. Policy windows, fees and phone numbers below were
accurate for US consumer accounts at compile time and change frequently — alwa

#### Step 2: Basic info

In [10]:
Total_pages=len(pages)
Total_characters=sum(len(page["text"])for page in pages)
print(f"Total number of pages :{Total_pages}")
print(f"Total Chracters :{Total_characters}")
print(f"First 500 chracters \n:{pages[0]["text"][:500]}")

Total number of pages :134
Total Chracters :328192
First 500 chracters 
:AMAZON.COM — HELP CENTRE
Amazon Support
Knowledge Base
Orders • Delivery • Returns • Refunds • Payments • Prime • Account & Privacy
20 sections · 359 answered entries · 22 reference tables · worked case files · letter templates · topic index
Search the help centre:  where is my order  |  return window  |  refund not received  |  a-to-z claim
 |  prime renewal
Edition
Compiled edition 2026.09 — Rev. 3.0
Coverage
US consumer and business accounts, with an India comparison in Appendix C and a
selle


### Step 3: Data Cleaning

In [11]:
import re
cleaned_pages=[]
for page in pages:
    page_text=page["text"]

    # Replacing multiple spaces and tabs with a single space
    page_text=re.sub(r"[\t]+"," ",page_text)

    # removing the excessive blank lines
    page_text=re.sub(r"\n{3,}","\n\n",page_text)

    # Remove the leading and trailing spaces
    page_text=page_text.strip()

    cleaned_pages.append({
        "page":page["page"],
        "text":page_text
    })
print(f"pages cleaned :{len(cleaned_pages)}")
print(f"First 500 characters \n :{cleaned_pages[0]["text"][:500]}")


pages cleaned :134
First 500 characters 
 :AMAZON.COM — HELP CENTRE
Amazon Support
Knowledge Base
Orders • Delivery • Returns • Refunds • Payments • Prime • Account & Privacy
20 sections · 359 answered entries · 22 reference tables · worked case files · letter templates · topic index
Search the help centre:  where is my order  |  return window  |  refund not received  |  a-to-z claim
 |  prime renewal
Edition
Compiled edition 2026.09 — Rev. 3.0
Coverage
US consumer and business accounts, with an India comparison in Appendix C and a
selle


### Step 4: Creating chunking

In [ ]:
def fixed_size_chunk(cleaned_pages, chunk_size=1000, overlap=200):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks = []
    for page in cleaned_pages:
        page_text = page["text"]
        page_num  = page["page"]
        start = 0
        while start < len(page_text):
            end = start + chunk_size
            chunk_text = page_text[start:end].strip()
            if chunk_text:
                chunks.append({
                    "page": page_num,
                    "text": chunk_text,
                })
            start += chunk_size - overlap
    return chunks
chunks=fixed_size_chunk(cleaned_pages)
print(f"Created {len(chunks)} chunks")

Created 476 chunks


### Step 4: Validate chunk

In [26]:
print("Total chunks:", len(chunks))

chunk_lengths = [len(chunk["text"]) for chunk in chunks]

print("Shortest chunk:", min(chunk_lengths))
print("Longest chunk:", max(chunk_lengths))
print("Average chunk length:", round(sum(chunk_lengths) / len(chunk_lengths), 2))

# Inspect the first 3 chunks
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {i} | Page {chunk['page']} ---")
    print(chunk["text"])

Total chunks: 476
Shortest chunk: 3
Longest chunk: 1000
Average chunk length: 826.07

--- Chunk 1 | Page 1 ---
AMAZON.COM — HELP CENTRE
Amazon Support
Knowledge Base
Orders • Delivery • Returns • Refunds • Payments • Prime • Account & Privacy
20 sections · 359 answered entries · 22 reference tables · worked case files · letter templates · topic index
Search the help centre:  where is my order  |  return window  |  refund not received  |  a-to-z claim
 |  prime renewal
Edition
Compiled edition 2026.09 — Rev. 3.0
Coverage
US consumer and business accounts, with an India comparison in Appendix C and a
seller-side section
Compiled
Last reviewed: 30 September 2026
Classification
Desk reference — buyer, seller and support trainee edition
AMZ-KB-2026-09 · REV 3.0 · 20 SECTIONS · 359 ENTRIES
NOTICE. This is an independently compiled reference guide written in the style of an e-commerce help centre. It is not
produced, reviewed or endorsed by Amazon.com, Inc. or any affiliate. Policy windows, f

### Step 6: Generating embeddings to chunks

In [33]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_text=[chunk["text"] for chunk in chunks]
embeddings=model.encode(
    chunk_text,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
print(f"Embeddings shape :{embeddings.shape}")
assert len(embeddings) == len(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings shape :(476, 384)


In [35]:
assert len(embeddings) == len(chunks)
print("Embedding validation passed!")

Embedding validation passed!


### Step 7: Saving embeddings to Faiss

In [36]:
import faiss
import numpy as np

embedding_dimension = embeddings.shape[1]
support_index = faiss.IndexFlatIP(embedding_dimension)
support_index.add(
    np.asarray(embeddings, dtype="float32")
)

print("Total vectors in FAISS:", support_index.ntotal)

Total vectors in FAISS: 476


### step 8: Save the FAISS index and chunk metadata

In [37]:
import json
from pathlib import Path
support_vector_dir = Path("vector_store")
support_vector_dir.mkdir(parents=True, exist_ok=True)

faiss.write_index(
    support_index,
    str(support_vector_dir / "support_index.faiss")
)

with open(
    support_vector_dir / "support_chunks.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

print("Support FAISS index and chunks saved successfully!")

Support FAISS index and chunks saved successfully!


### Step 9: Making retrive Function

In [39]:
def retrieve_support_chunks(query, top_k=5):
    query_embedding =model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )
    scores, indices = support_index.search(
        query_embedding,
        top_k
    )
    retrieved_chunks = []
    for score, index_position in zip(scores[0], indices[0]):
        if index_position == -1:
            continue
        chunk = chunks[index_position].copy()
        chunk["similarity_score"] = float(score)
        retrieved_chunks.append(chunk)
    return retrieved_chunks
print("Done")

Done


#### Test

In [40]:
query = "How can I return a product?"
results = retrieve_support_chunks(query, top_k=5)
for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Page:", result["page"])
    print("Similarity:", round(result["similarity_score"], 4))
    print(result["text"])


--- Result 1 ---
Page: 5
Similarity: 0.5727
AMAZON SUPPORT KNOWLEDGE BASE
AMZ-KB-2026-09 · Rev 3.0
Page 5 of 134
1
0
What happens to my item at the returns facility?
1
1
Can someone else drop off my return, and how do gift returns work?
1
2
What is a “return rate” and can it get my returns restricted?
1
3
Can I return an opened subscription-box item?
1
4
The return label never arrived or will not print. What now?
1
5
Can I drop a return off at a physical store without a box?
1
6
If the price of an item I returned was higher than what I paid, do I get the difference?
1
7
My cart said an item was returnable and now the order page says otherwise. Which wins?
1
8
The carrier lost my return in transit. Am I liable?
1
9
Do return shipping charges ever get waived retroactively?
2
0
Can I return a personalised or made-to-order item?
2
1
How do returns work for large items like furniture and appliances?
06
Refunds
p. 45
1
What does “refund issued” mean, and when will the money appear?
2
My ref

### Step 10 : Connect LLM layer

In [41]:
from src.llm import ask_amazon
def support_rag_answer(query, top_k=5):
    retrieved_chunks = retrieve_support_chunks(
        query,
        top_k=top_k
    )
    context = "\n\n".join(
        [
            f"Source: Page {chunk['page']}\n{chunk['text']}"
            for chunk in retrieved_chunks
        ]
    )
    answer = ask_amazon(query, context)
    return answer
print("Done")

Done All works perfectly
Done


In [43]:
query = "How can I return a product?"
answer = support_rag_answer(query)
print("Question:", query)
print("\nAnswer:\n", answer)

Question: How can I return a product?

Answer:
 To return a product, follow these steps:

1. **Start the return**  
   • Go to your Amazon order page.  
   • Select the item you want to return and click **“Return or replace items.”**  
   • Choose the reason for the return and confirm.

2. **Print or obtain a return label**  
   • Amazon will generate a prepaid return label.  
   • Print the label and attach it to the package.  
   • If the label doesn’t print or never arrives, you can:  
     – Use the QR‑code at a partner store (Whole Foods, Amazon Fresh, many UPS or Kohl’s locations).  
     – Ask Amazon Support to re‑issue a label.

3. **Prepare the item**  
   • Use the original packaging and accessories if possible.  
   • If you don’t have the box, many stores accept loose items with the QR‑code.

4. **Drop it off**  
   • Take the package to a UPS drop‑off location, a partner store, or an Amazon return center.  
   • Keep the receipt or proof of drop‑off.

5. **Track the return